In [32]:
# Header for the notebook
from datetime import datetime
from IPython.display import display, Markdown

# Get the current date
title = "Movement smoothness as a subclinical marker in low back pain - is whole signal analysis pertinent? "
current_date = datetime.now().strftime("%d %B %Y, %H:%M:%S")
authors = "Ancelin Gely (and Copilot)"

# Insert the date into the notebook
display(Markdown(f"# {title}"))
display(Markdown(f"{current_date}"))
display(Markdown(f"by {authors}"))

# Movement smoothness as a subclinical marker in low back pain - is whole signal analysis pertinent? 

09 May 2026, 11:08:50

by Ancelin Gely (and Copilot)

# Python Project : Load, filter and analyze data 

## Aim of the code 

The aim of this code is to extract the velocity profile from the gyroscope data, segment the movement into repetition of flexion - return from flexion (extension) and compute the SPARC and NNP for flexion and for the whole movement. 

## Data organization

### File name 
Flexion_Dos_OOX_av.xlsx
X is the patient ID (1-10)

### File structure
The data is organized as a table with different sheets : 
- general informations 
- Markers
- Segment orientation - Quat
- Segment orientation - Euler
- Segment position 
- Segment velocity
- Segment acceleration
- Segment angular velocity
- Ergonomic joint angle 
- Center of mass
- Sensor free acceleration
- Sensor magnetic field
- Sensor orientation - Quat
- Sensor orientation - Euler

### Gyroscope data
The gyroscope data is located in the sheet "Segment angular velocity". It contains the angular velocity of the segment in three dimensions (x, y, z). 

Columns :
- Frame	
- Pelvis x, y, z		
- L5 x, y, z
- L3 x, y, z
- T12 x, y, z	
- T8 x, y, z

## Notebook organization

In this jupiter notebook, we will perform the following steps :
1. Load the data and filter the gyroscope data for L3 Y (L3)
2. Segment the movement into repetitions of flexion return from flexion (extension)
3. Compute the SPARC and NNP for each repetition and for the whole movement
4. Add the results to a dataframe and save it as a xlsx file


### Compute SPARC and NNP 

#### Spectral arc lenght (SPARC) : 

The spectral arc lenght (SPARC) is a frequency-based metric that quantifies the smoothness of a movement by analyzing the Fourier magnitude spectrum of the velocity profile. The velocity profile is transformed into the frequency domain using the Fourier transform, and the SPARC is calculated based on the lenght of the arc. This lenght is calculated by integrating the euclidean distance between the points of the Fourier magnitude spectrum. A smooth movement will have a regular and simple sperctra while an irregular movement will have a more complex spectrum with more hight frequencies. SPARC values typically range from 0 to -∞. A SPARC value of 0 indicates a perfectly smooth movement, while a SPARC value of -∞ indicates a highly irregular movement (4, copilote)

#### Number of peaks (NNP) :

The number of peaks (NNP) is a time-domain metric that quantifies the smoothness of a movement by counting the number of peaks in the velocity profile. The result is then normalized by the movement duration. NNP values typically range from 0 to +∞, with lower values indicating smoother movements. A NNP value of 0 indicates a perfectly smooth movement, while higher NNP values indicate more irregular movements with more peaks in the velocity profile (copilote).

**More details on the SPARC and NNP computation can be found in the parts "SPARC computation" and "NNP computation" of the code**


# Code 

## Import libraries



In [33]:
# If you need to install the libraries, uncomment the following lines and run them once. After that, you can comment them again.
# %pip install numpy
# %pip install pandas
# %pip install matplotlib
# %pip install scipy
# %pip install IPython
# %pip install openpyxl

# Import library
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

from scipy.signal import butter
from scipy.signal import filtfilt
from scipy.signal import find_peaks

### Load the data and filter the gyroscope data for L3 Y (L3)

#### Load and reshape the data
The data is loaded from the excel file using pandas and the gyroscope data for L3 Y is extracted. Frame to time conversion is necessary. So the time is calculated by dividing the frame number by the sampling frequency (60 Hz).

$$time = \frac{frame}{\text{sampling frequency}}$$

#### Filter the signal
A low pass Butterworth filter is applied to the data to obtain a smoother velocity profile. 

The filter is composed of (copilote): 

- order : 2 = determines the steepness of the filter. An order of 2 provides a moderate roll-off, which is suitable for removing high-frequency noise while preserving the main characteristics of the movement signal.

- cutoff frequency : 10 Hz = determines the frequency at which the filter starts to attenuate the signal. A cutoff frequency of 10 Hz is appropriate for human movement analysis, as it allows to retain the relevant movement information while filtering out high-frequency noise.

- fc/(fs/2) : the cutoff frequency is normalized by the Nyquist frequency (half of the sampling frequency) to ensure that the filter is designed correctly for the given sampling rate.

- b, a : the filter coefficients are calculated using the Butterworth filter design function from the scipy library. These coefficients are then used to apply the filter to the signal using the filtfilt function, which applies the filter forward and backward to avoid phase distortion.

- low pass : the type of filter is specified as low pass, which means that it allows frequencies below the cutoff frequency to pass through while attenuating frequencies above the cutoff frequency. 

In [34]:
#######################################
######## LOAD + FILTER ################
#######################################

def load_and_filter_signal(file_path, sheet_name, column_name, fs, fc):

    signal = pd.read_excel(file_path, sheet_name=sheet_name)

    # Frame to time conversion
    if "Time (s)" not in signal.columns:
        if "Frame" not in signal.columns:
            raise ValueError("Time (s) or Frame column missing")
        signal["Time (s)"] = signal["Frame"] / fs

    omega = signal[column_name].to_numpy()

    # Low-pass filter
    b, a = butter(2, fc / (fs / 2), btype="low")
    omega_filt = filtfilt(b, a, omega)

    return signal, omega_filt

### Segment detection
Patients were asked to perform 5 repetitions of flexion-extension, so the signal is composed of 5 segments of flexion and 5 segments of return from flexion (extension).

The movement is segmented into repetitions of flexion-extension by transforming the velocity profile into a binary signal using a threshold value. The threshold is set to 0.01 to ensure that only significant movements are detected, while small fluctuations in the signal are ignored. The binary signal is created as follows :

- 1 : Values above the threshold
- 0 : Values below the threshold

Then, the difference between the binary signal is calculated to identify the start and end of each repetition. The difference is computed as follows :

- 1 (1-0) : transition from extension to flexion, indicating the start of a flexion phase
- -1 (0-1) : transition from flexion to extension, indicating the end of the flexion phase
- 0 : no transition. 

The start and end of each repetition are then associated with peaks in the velocity profile to ensure that the segmentation corresponds to actual movement phases. The find_peaks function from the scipy library is used to identify the peaks in the velocity profile, which correspond to the maximum velocity during each flexion and extension phase. A segment correspond to a period where the peak value is located between the start and end points. Segments shorter than 0.5 seconds are then excluded from the analysis to ensure that only real repetitions are considered. Consecutive segments of the same type (flexion or extension) are suppressed to avoid counting artifacts. 


In [35]:
#######################################
######## SEGMENT DETECTION ############
#######################################

def detect_segments(omega_filt, eps, fs, min_peak_distance, signal):

    # Negative segments
    neg_mask = omega_filt < -eps
    neg_transitions = np.diff(neg_mask.astype(int))

    neg_starts = np.where(neg_transitions == 1)[0] + 1
    neg_ends = np.where(neg_transitions == -1)[0] + 1

    if neg_mask[0]:
        neg_starts = np.insert(neg_starts, 0, 0)

    if neg_mask[-1]:
        neg_ends = np.append(neg_ends, len(neg_mask))

    # Positive segments
    pos_mask = omega_filt > eps
    pos_transitions = np.diff(pos_mask.astype(int))

    pos_starts = np.where(pos_transitions == 1)[0] + 1
    pos_ends = np.where(pos_transitions == -1)[0] + 1

    if pos_mask[0]:
        pos_starts = np.insert(pos_starts, 0, 0)

    if pos_mask[-1]:
        pos_ends = np.append(pos_ends, len(pos_mask))

    # Peak detection
    positive_peaks, _ = find_peaks(
        omega_filt,
        prominence=0.9 * np.std(omega_filt),
        distance=int(min_peak_distance * fs)
    )

    negative_peaks, _ = find_peaks(
        -omega_filt,
        prominence=0.9 * np.std(omega_filt),
        distance=int(min_peak_distance * fs)
    )

    # Associate peaks with segments
    negative_segments = []

    for p in negative_peaks:
        idx = np.where((neg_starts <= p) & (neg_ends >= p))[0]
        if len(idx) == 1:
            negative_segments.append(
                (int(neg_starts[idx[0]]), int(neg_ends[idx[0]]))
            )

    positive_segments = []

    for p in positive_peaks:
        idx = np.where((pos_starts <= p) & (pos_ends >= p))[0]
        if len(idx) == 1:
            positive_segments.append(
                (int(pos_starts[idx[0]]), int(pos_ends[idx[0]]))
            )

    # Remove segments shorter than 0.5 s
    negative_segments = [
        (s, e) for s, e in negative_segments
        if (signal["Time (s)"].iloc[e - 1] - signal["Time (s)"].iloc[s]) >= 0.5
    ]

    positive_segments = [
        (s, e) for s, e in positive_segments
        if (signal["Time (s)"].iloc[e - 1] - signal["Time (s)"].iloc[s]) >= 0.5
    ]

    # Suppress consecutive segments
    def suppress_consecutive_segments(segments):

        if not segments:
            return []

        suppressed = [segments[0]]
        for s, e in segments[1:]:
            last_s, last_e = suppressed[-1]
            if (s - last_e) > 1:
                suppressed.append((s, e))

        return suppressed

    negative_segments = suppress_consecutive_segments(negative_segments)
    positive_segments = suppress_consecutive_segments(positive_segments)

    return positive_segments, negative_segments

### SPARC computation

The continue formulation of the SPARC is given by the following formula. It is based on the formula of an arc lenght where (1/wc).dw permits to normalize on the frequency range which allow the comparision between different signals.

$$SPARC = - ∫ |\sqrt (\frac{1}{wc}+\frac{dV(ω)}{dω}) dω |$$

But in practice, the SPARC is computed using a discrete approximation of the integral, which involves summing the euclidean distance between the points of the Fourier magnitude spectrum at discrete frequency points. The discrete approximation is given by the following formula:

$$SPARC = - \sum |\sqrt(ds^2 + df^2)|$$

Where ds and df are the discrete differences of the Fourier magnitude spectrum and the frequency, respectively.

**df computation :** (copilote)
1. Compute the Fourier magnitude spectrum of the velocity profile using the Fourier transform. 
2. Normalize the Fourier magnitude spectrum by dividing it by its maximum value to ensure that the SPARC value is independent of the signal amplitude.
3. Apply a log transformation to the normalized Fourier magnitude spectrum to reduce the influence of large values and enhance the contribution of smaller values, which can provide a more accurate representation of the movement smoothness.
4. Calculate the derivative of the Fourier magnitude spectrum with respect to frequency to obtain the rate of change of the spectrum across frequencies. This derivative captures how the energy distribution in the frequency domain changes, which is essential for quantifying the smoothness of the movement.

**ds computation :** (copilote)
- Compute the frequency range over which the SPARC is calculated. This range is typically defined based on the sampling frequency and the length of the velocity profile, and it determines the frequencies that are included in the SPARC calculation. The frequency range is often limited to a specific range (e.g., 0 to 10 Hz) to focus on the relevant frequencies for human movement analysis and to exclude high-frequency noise that may not contribute to the smoothness of the movement.

**arc length computation :** (copilote)
- Compute the arc length of the Fourier magnitude spectrum by integrating the euclidean distance between the points of the spectrum across the defined frequency range. The arc length is calculated by summing the square root of the sum of squares of ds and df, which represents the distance between consecutive points in the frequency domain. This integration captures the overall shape and complexity of the spectrum, which is indicative of the smoothness of the movement.


In [36]:
#######################################
############ COMPUTE SPARC ############
#######################################

def compute_sparc(x, fs):

    x = x - np.mean(x)
    n = len(x)
    threshold = 0.05

    # Compute ds
    spectrum = np.abs(np.fft.rfft(x))
    max_spectrum = np.max(spectrum)
    if max_spectrum == 0:
        return np.nan

    spectrum = spectrum / max_spectrum
    spectrum += 1e-10
    valid = spectrum > threshold
    spectrum = spectrum[valid]
    ds = np.diff(spectrum)

    # compute df
    freqs = np.fft.rfftfreq(n, d=1/fs)
    freqs = freqs[valid]
    df = np.diff(freqs)
    
    # compute arc length
    arc_length = np.sum(np.sqrt(df**2 + ds**2))

    return -arc_length

### NNP computation

The number of peaks (NNP) is computed by counting the number of peaks in the velocity profile using the find_peaks function from the scipy library. The prominence parameter is set to 0.005 to ensure that only significant peaks are counted, while small fluctuations in the signal are ignored. The number of peaks is then normalized by the movement duration to account for differences in movement speed. 

$$NNP = \frac{\text{number of peaks}}{\text{movement duration}}$$

In [37]:
def compute_nnp(signal, time, prominence=0.005):

    peaks, _ = find_peaks(signal, prominence=prominence)
    duration = time.iloc[-1] - time.iloc[0]

    if duration <= 0:
        return np.nan

    nnp = len(peaks) / duration

    return nnp

### analyse_signal_v2 computation
The analyse_signal_v2 function performs the following steps:
1. Load the data and filter the gyroscope data for L3 Y (L3)
2. Segment the movement into repetitions of flexion return from flexion (extension)
3. Compute the SPARC and NNP for each flexion and for the whole movement
4. Add the results to a dataframe. 

In [38]:
def analyze_signal_v2(file_path, sheet_name, column_name):

    # Parameters
    fs = 60
    fc = 10
    eps = 0.01
    min_peak_distance = 0.5

    ###################################
    ######## LOAD + FILTER ############
    ###################################

    signal, omega_filt = load_and_filter_signal(
        file_path,
        sheet_name,
        column_name,
        fs,
        fc
    )

    ###################################
    ######## SEGMENTATION #############
    ###################################

    positive_segments, negative_segments = detect_segments(
        omega_filt,
        eps,
        fs,
        min_peak_distance,
        signal
    )

    ###################################
    ########### SPARC #################
    ###################################

    # Compute SPARC for flexion segments
    positive_sparc = []
    for start, end in positive_segments:

        positive_sparc.append(
            compute_sparc(omega_filt[start:end], fs)
        )

    # Compute SPARC for the whole signal
    SPARC = compute_sparc(omega_filt, fs)

    ###################################
    ############ NNP ##################
    ###################################

    # Compute NNP for flexion segments
    flexion_peak_counts = []

    for s, e in positive_segments:
        segment_omega = omega_filt[s:e]
        segment_time = signal["Time (s)"].iloc[s:e]
        flexion_peak_counts.append(
            compute_nnp(segment_omega,segment_time)
        )

    # Whole signal NNP
    nnp = compute_nnp(omega_filt, signal["Time (s)"])


    ###################################
    ######## RESULTS TABLE ############
    ###################################

    signal_name = file_path.split("/")[-1].split(".")[0]

    results = pd.DataFrame({
        "Signal name": [signal_name],
        "SPARC": [SPARC],
        "NNP": [nnp],
        "mean_SPARC_flexion": [np.nanmean(positive_sparc)],
        "mean_NNP_flexion": [np.nanmean(flexion_peak_counts)]
    })

    return results

# Analyse signals from all patients
As the folder data containes data from 10 patients called Flexion_Dos_OOX_av.xlsx, where X is the patient ID (1-10), we can loop through all the files and apply the analyse_signal_v2 function to each file to extract the SPARC and NNP values for each patient. The results can then be compiled into a single dataframe for further analysis and comparison across patients.


In [39]:
# Create a loop to analyze all the signals and concatenate the results in a single dataframe.
results_all = pd.DataFrame()
for i in range(1, 11):
    file_number = f"{i:03d}"
    file_path = f"data/FlexionDos_{file_number}_av.xlsx"
    sheet_name = "Segment Angular Velocity"
    column_name = "L3 y"
    result = analyze_signal_v2(file_path, sheet_name, column_name)
    results_all = pd.concat([results_all, result], ignore_index=True)

# Results 

In [40]:
results_all

,Signal name,SPARC,NNP,mean_SPARC_flexion,mean_NNP_flexion
0,FlexionDos_001_av,-2.108382,1.670103,-1.898486,1.804946
1,FlexionDos_002_av,-2.454168,3.946588,-3.126696,3.452343
2,FlexionDos_003_av,-3.781590,3.625192,-3.931707,4.445737
3,FlexionDos_004_av,-2.674619,1.224490,-1.572919,1.109087
4,FlexionDos_005_av,-3.856549,5.095796,-9.013787,5.819458
5,FlexionDos_006_av,-3.063880,2.743022,-1.679951,2.559037
6,FlexionDos_007_av,-2.568564,1.015572,-1.757691,0.961883
7,FlexionDos_008_av,-2.872345,3.790782,-4.745600,3.924780
8,FlexionDos_009_av,-2.625414,1.893048,-1.561279,2.456543
9,FlexionDos_010_av,-2.871953,4.101877,-4.213141,4.360160


In [41]:
# Save as xlsx in the folder "results"
results_all.to_excel("results/results_all.xlsx", index=False)
